# Fongbe ASR Training - Google Colab

Fine-tuning Whisper for Fongbe speech recognition using LoRA.

## Prerequisites
1. Runtime → GPU (T4 recommended)
2. Public datasets are downloaded automatically to `MyDrive/fongbe/data/raw`

## 1. Setup Environment

In [1]:
from pathlib import Path
from google.colab import drive
import subprocess

# Mount Drive
drive.mount('/content/drive')

# Config paths
PROJECT_ROOT = Path('/content/drive/MyDrive/fongbe')
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
DATA_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'fongbe_asr_unified'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
REPO_URL = 'https://github.com/Appolinairee/fongbe-asr.git'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"✓ Project: {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Project: /content/drive/MyDrive/fongbe


## 2. Clone Repository

In [2]:
import os

os.chdir('/content')
if not Path('fongbe-asr').exists():
    !git clone {REPO_URL} fongbe-asr
else:
    !cd fongbe-asr && git pull

os.chdir('fongbe-asr')
print(f"✓ Working dir: {Path.cwd()}")

Cloning into 'fongbe-asr'...
remote: Enumerating objects: 14512, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 14512 (delta 17), reused 35 (delta 9), pack-reused 14467 (from 1)
Receiving objects: 100% (14512/14512), 45.13 MiB | 25.71 MiB/s, done.
Resolving deltas: 100% (3311/3311), done.
✓ Working dir: /content/fongbe-asr


## 3. Download and Prepare Dataset

In [3]:
import os

if not (DATA_ROOT / 'train').exists():
    print("Downloading public sources to Drive...")
    !python scripts/download_sources_to_drive.py --dest "{RAW_ROOT}"

    print("Preparing unified training dataset...")
    os.environ['RAW_DATA_DIR'] = str(RAW_ROOT)
    os.environ['DATASET_PATH'] = str(DATA_ROOT)
    !python scripts/prepare_dataset_hf.py
else:
    print("✓ Dataset already prepared")

# Verify
n_train = len(list((DATA_ROOT / 'train').glob('*.arrow')))
print(f"✓ {n_train} files in train/")

✓ Dataset already extracted
✓ 1 files in train/


## 4. Install Dependencies

In [4]:
!pip install -q transformers accelerate peft evaluate jiwer datasets tensorboard soundfile librosa
print("✓ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 43.6 MB/s eta 0:00:0000:0100:01
✓ Dependencies installed


In [5]:
# Fix torchao version
!pip install -q --upgrade torchao
print("✓ torchao upgraded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 37.6 MB/s eta 0:00:00a 0:00:01
✓ torchao upgraded


## 5. Verify GPU

In [6]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Runtime → Change runtime type → GPU")

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


## 6. Run Training

In [7]:
# Set paths for training script
import os
os.environ['DATASET_PATH'] = str(DATA_ROOT)
os.environ['OUTPUT_DIR'] = str(OUTPUT_ROOT / 'whisper-fongbe')

!python scripts/finetune_whisper.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
🔧 FINETUNING WHISPER FONGBE
Modèle: openai/whisper-small
LoRA rank: 8
Target modules: ['q_proj', 'v_proj']

📦 Chargement dataset...
✅ Dataset chargé:
   Train: 10864 samples
   Validation: 1358 samples
   Test: 1359 samples

🤖 Chargement Whisper...
preprocessor_config.json: 100% 185k/185k [00:00<00:00, 218MB/s]
config.json: 100% 1.97k/1.97k [00:00<00:00, 6.18MB/s]
tokenizer_config.json: 100% 283k/283k [00:00<00:00, 209MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 25.1MB/s]
tokenizer.json: 100% 2.48M/2.48M [00:00<00:00, 148MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 96.2MB/s]
norm

## 7. TensorBoard (Optional)

In [19]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_ROOT / 'whisper-fongbe'}

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 4502), started 0:03:40 ago. (Use '!kill 4502' to kill it.)

<IPython.core.display.Javascript object>